In [ ]:
%%writefile lab5.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

/* Структура заявки */
typedef struct {
    int  id;
    char description[100];
    char status[50];
    char date[20];
} Request;

/* Узел дерева */
typedef struct Node {
    Request      data;
    struct Node *left;
    struct Node *right;
} Node;

/* Узел очереди для BFS */
typedef struct QNode {
    Node        *treeNode;
    struct QNode *next;
} QNode;

/* Очередь */
typedef struct {
    QNode *front;
    QNode *rear;
} Queue;

/* ---- Очередь ---- */

Queue *createQueue() {
    Queue *q = malloc(sizeof(Queue));
    q->front = q->rear = NULL;
    return q;
}

void enqueue(Queue *q, Node *n) {
    QNode *qn = malloc(sizeof(QNode));
    qn->treeNode = n;
    qn->next = NULL;
    if (q->rear) q->rear->next = qn;
    else q->front = qn;
    q->rear = qn;
}

Node *dequeue(Queue *q) {
    if (!q->front) return NULL;
    QNode *tmp = q->front;
    Node  *n   = tmp->treeNode;
    q->front   = tmp->next;
    if (!q->front) q->rear = NULL;
    free(tmp);
    return n;
}

int isQueueEmpty(Queue *q) {
    return q->front == NULL;
}

/* ---- Ввод / вывод ---- */

static void clean_newline(char *s) {
    size_t len = strlen(s);
    if (len > 0 && s[len-1] == '\n') s[len-1] = '\0';
}

static void read_line(const char *prompt, char *buf, int n) {
    printf("%s", prompt);
    fflush(stdout);
    if (fgets(buf, n, stdin)) clean_newline(buf);
    else buf[0] = '\0';
}

static void clear_buffer() {
    int c;
    while ((c = getchar()) != '\n' && c != EOF);
}

void readRequest(Request *r) {
    printf("Введите ID: ");
    scanf("%d", &r->id);
    clear_buffer();
    read_line("Описание проблемы: ",           r->description, 100);
    read_line("Статус (в обработке/завершено): ", r->status,   50);
    read_line("Дата подачи (ДД.ММ.ГГГГ): ",    r->date,       20);
}

void printRequest(const Request *r) {
    printf("  ID: %d | %s | %s | %s\n",
           r->id, r->description, r->status, r->date);
}

/* ---- Дерево ---- */

Node *createNode(Request r) {
    Node *n  = malloc(sizeof(Node));
    n->data  = r;
    n->left  = n->right = NULL;
    return n;
}

/* Вставка по ID (BST: меньше — влево, больше — вправо) */
Node *insert(Node *root, Request r) {
    if (!root) return createNode(r);
    if (r.id < root->data.id)
        root->left  = insert(root->left,  r);
    else if (r.id > root->data.id)
        root->right = insert(root->right, r);
    else
        printf("Заявка с ID=%d уже существует.\n", r.id);
    return root;
}

/* Поиск по ID */
Node *search(Node *root, int id) {
    if (!root)            return NULL;
    if (id == root->data.id) return root;
    if (id < root->data.id)  return search(root->left,  id);
                              return search(root->right, id);
}

void freeTree(Node *root) {
    if (!root) return;
    freeTree(root->left);
    freeTree(root->right);
    free(root);
}

/* ---- Обходы ---- */

/* Прямой: NLR */
void preorder(Node *root) {
    if (!root) return;
    printRequest(&root->data);
    preorder(root->left);
    preorder(root->right);
}

/* Центрированный: LNR (он же DFS) */
void inorder(Node *root) {
    if (!root) return;
    inorder(root->left);
    printRequest(&root->data);
    inorder(root->right);
}

/* Обратный: LRN */
void postorder(Node *root) {
    if (!root) return;
    postorder(root->left);
    postorder(root->right);
    printRequest(&root->data);
}

/* Обход в ширину: BFS */
void bfs(Node *root) {
    if (!root) return;
    Queue *q = createQueue();
    enqueue(q, root);
    while (!isQueueEmpty(q)) {
        Node *cur = dequeue(q);
        printRequest(&cur->data);
        if (cur->left)  enqueue(q, cur->left);
        if (cur->right) enqueue(q, cur->right);
    }
    free(q);
}

/* ---- Главное меню ---- */

int main() {
    Node *root = NULL;
    int choice, id;

    do {
        printf("\n1. Добавить заявку\n");
        printf("2. Прямой обход (NLR)\n");
        printf("3. Центрированный обход / DFS (LNR)\n");
        printf("4. Обратный обход (LRN)\n");
        printf("5. Обход в ширину (BFS)\n");
        printf("6. Поиск по ID\n");
        printf("0. Выход\n");
        printf("Выбор: ");
        if (scanf("%d", &choice) != 1) break;
        clear_buffer();

        switch (choice) {
            case 1: {
                Request r;
                readRequest(&r);
                root = insert(root, r);
                printf("Заявка добавлена.\n");
                break;
            }
            case 2:
                printf("\n--- Прямой обход (NLR) ---\n");
                if (!root) printf("Дерево пусто.\n");
                else preorder(root);
                break;
            case 3:
                printf("\n--- Центрированный обход / DFS (LNR) ---\n");
                if (!root) printf("Дерево пусто.\n");
                else inorder(root);
                break;
            case 4:
                printf("\n--- Обратный обход (LRN) ---\n");
                if (!root) printf("Дерево пусто.\n");
                else postorder(root);
                break;
            case 5:
                printf("\n--- Обход в ширину (BFS) ---\n");
                if (!root) printf("Дерево пусто.\n");
                else bfs(root);
                break;
            case 6:
                printf("Введите ID для поиска: ");
                scanf("%d", &id);
                clear_buffer();
                Node *found = search(root, id);
                if (found) {
                    printf("Найдено:\n");
                    printRequest(&found->data);
                } else {
                    printf("Заявка с ID=%d не найдена.\n", id);
                }
                break;
            case 0:
                break;
            default:
                printf("Неверный выбор.\n");
        }
    } while (choice != 0);

    freeTree(root);
    return 0;
}

Writing lab5.c


In [ ]:
!gcc lab5.c -o lab5 -Wall
!./lab5


1. Добавить заявку
2. Прямой обход (NLR)
3. Центрированный обход / DFS (LNR)
4. Обратный обход (LRN)
5. Обход в ширину (BFS)
6. Поиск по ID
0. Выход
Выбор: 1
Введите ID: 423
Описание проблемы: не работает плата
Статус (в обработке/завершено): в обработке
Дата подачи (ДД.ММ.ГГГГ): 28.03.2026
Заявка добавлена.

1. Добавить заявку
2. Прямой обход (NLR)
3. Центрированный обход / DFS (LNR)
4. Обратный обход (LRN)
5. Обход в ширину (BFS)
6. Поиск по ID
0. Выход
Выбор: 1
Введите ID: 222
Описание проблемы: проблемы з блоком питания
Статус (в обработке/завершено): завершено
Дата подачи (ДД.ММ.ГГГГ): 29.03.2026
Заявка добавлена.

1. Добавить заявку
2. Прямой обход (NLR)
3. Центрированный обход / DFS (LNR)
4. Обратный обход (LRN)
5. Обход в ширину (BFS)
6. Поиск по ID
0. Выход
Выбор: 6
Введите ID для поиска: 222
Найдено:
  ID: 222 | проблемы з блоком питания | завершено | 29.03.2026

1. Добавить заявку
2. Прямой обход (NLR)
3. Центрированный обход / DFS (LNR)
4. Обратный обход (LRN)
5. Обход в ш